# Part 15: A Tiny Eval Harness

*Evals from First Principles, Part 15 (capstone).*

Every part in this series built one reusable piece: a metric (Part 1), an LLM judge (Part 4), a bootstrap confidence interval (Part 6), a release gate that reads the CI floor instead of the point estimate (Part 11), and a trajectory scorer for agents (Part 14). This capstone wires all five into one small, offline, pure-Python harness, then runs it three ways on two tiny datasets: a RAG-style QA set scored first with strict exact match and then with a lenient mock judge, and an agent tool-call trajectory set scored with step-level credit. Swap the dataset or the scorer; everything downstream, the CI, the gate, the printout, stays unchanged.

In [ ]:
import numpy as np

## Piece 1: the datasets

Two tiny, hand-labeled sets, small enough to print every row so every number stays traceable back to a specific item.

The QA set is 10 capital-cities questions (a nod to RAG Part 8): a gold short answer and the system's candidate. Item 6 is the interesting one: `"Washington, D.C."` is a correct answer for `"Washington"`, but a strict string match will unfairly fail it. The judge (piece 2) will catch that.

The trajectory set is 5 agent tasks (a nod to Part 14 / Agents Part 17): for each, the gold sequence of tool calls and the agent's actual sequence.

In [ ]:
# A RAG-style QA set: a question, the GOLD short answer, the system's CANDIDATE.
# Item 6 is the interesting one: "Washington, D.C." is correct for the USA,
# but a strict string match will unfairly fail it. The judge will catch that.
QA = [
    {"q": "capital of France",    "gold": "Paris",     "cand": "Paris"},
    {"q": "capital of Japan",     "gold": "Tokyo",     "cand": "Tokyo"},
    {"q": "capital of Australia", "gold": "Canberra",  "cand": "Sydney"},
    {"q": "capital of Canada",    "gold": "Ottawa",    "cand": "Ottawa"},
    {"q": "capital of Brazil",    "gold": "Brasilia",  "cand": "Rio de Janeiro"},
    {"q": "capital of USA",       "gold": "Washington","cand": "Washington, D.C."},
    {"q": "capital of Turkey",    "gold": "Ankara",    "cand": "Ankara"},
    {"q": "capital of Spain",     "gold": "Madrid",    "cand": "Madrid"},
    {"q": "capital of India",     "gold": "New Delhi", "cand": "new delhi"},
    {"q": "capital of Italy",     "gold": "Rome",      "cand": "Milan"},
]

# An agent-trajectory set: for each task, the GOLD sequence of tool calls and
# the agent's ACTUAL sequence. We score how much of the path lined up (Part 14).
TRAJ = [
    {"task": "answer from a doc",   "gold": ["search", "read", "answer"],
                                     "pred": ["search", "read", "answer"]},
    {"task": "compute a total",     "gold": ["search", "calculator", "answer"],
                                     "pred": ["search", "weather", "answer"]},
    {"task": "summarize findings",  "gold": ["search", "read", "summarize", "answer"],
                                     "pred": ["search", "read", "answer"]},
    {"task": "quick lookup",        "gold": ["search", "answer"],
                                     "pred": ["search", "search", "answer"]},
    {"task": "schedule a meeting",  "gold": ["calendar", "email"],
                                     "pred": ["calendar", "email"]},
]

## Piece 2: scorers

A scorer maps one record to a score in `[0, 1]`. Three of them, from strictest to most structural:

- `exact_match` (Part 1): 1.0 only if the candidate equals the gold answer after lowercasing and stripping whitespace.
- `mock_judge` (Part 4): a deterministic stand-in for an LLM judge. Its rubric is transparent (correct if the gold answer is contained in the candidate or vice versa), lenient enough to accept `"Washington, D.C."` for `"Washington"`. A real judge would swap in behind a `generate()` call; the verdict stays 0/1.
- `trajectory_match` (Part 14): the fraction of positions where the gold and predicted tool-call sequences agree, penalized by any length mismatch.

In [ ]:
def _norm(s):
    """Lowercase + strip: the cheapest possible answer normalization."""
    return s.strip().lower()


def exact_match(rec):
    """Strict string metric: 1.0 only if candidate equals gold after _norm."""
    return 1.0 if _norm(rec["cand"]) == _norm(rec["gold"]) else 0.0


def mock_judge(rec):
    """A deterministic stand-in for an LLM judge (Part 4). Its transparent
    rubric: 'correct if the gold answer is contained in the candidate, or vice
    versa' -- lenient enough to accept 'Washington, D.C.' for 'Washington'.
    A real judge would swap in behind a generate() call; the verdict stays 0/1."""
    g, c = _norm(rec["gold"]), _norm(rec["cand"])
    return 1.0 if (g in c or c in g) else 0.0


def trajectory_match(rec):
    """Step-level credit (Part 14): the fraction of positions where the gold
    and predicted tool calls agree, penalized by any length mismatch."""
    gold, pred = rec["gold"], rec["pred"]
    matches = sum(1 for i in range(min(len(gold), len(pred))) if gold[i] == pred[i])
    return matches / max(len(gold), len(pred))

## Piece 3: the bootstrap CI (Part 6)

The only randomness in the whole harness, and it is seeded. Resample the per-item scores with replacement `B=2000` times, re-average each resample, and read the 2.5th / 97.5th percentiles of those means as the 95% CI. A fixed seed makes the interval reproducible on any machine.

In [ ]:
def bootstrap_ci(scores, rng, B=2000, alpha=0.05):
    """Resample the per-item scores WITH replacement B times, re-average each
    time, and read the 2.5 / 97.5 percentiles of those means. Returns (lo, hi)."""
    a = np.asarray(scores, dtype=float)
    n = len(a)
    means = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        means[b] = a[idx].mean()
    lo = np.percentile(means, 100 * alpha / 2)
    hi = np.percentile(means, 100 * (1 - alpha / 2))
    return lo, hi

## Piece 4: the gate (Part 11)

The whole point of the series in one comparison: do not ship on a point estimate. Ship only when the uncertainty band clears a release bar: `CI_lo >= bar`.

In [ ]:
def gate(ci_lo, bar):
    """The whole point of the series in one comparison: do NOT ship on a point
    estimate; ship only when the uncertainty band stays above the bar."""
    return "SHIP" if ci_lo >= bar else "HOLD"

## Piece 5: the harness

`run_eval` wires pieces 1-4 together: score every item, bootstrap a CI around the mean, apply the gate, and print one line per item so every number stays traceable. Two small `show` helpers format the QA rows and the trajectory rows differently.

In [ ]:
def run_eval(name, dataset, scorer, bar, show, seed=0, B=2000):
    """Score every item, bootstrap a CI around the mean, and apply the gate.
    `show` prints one line per item so every number stays traceable."""
    scores = [scorer(rec) for rec in dataset]
    mean = sum(scores) / len(scores)
    rng = np.random.default_rng(seed)      # fixed seed -> reproducible CI
    lo, hi = bootstrap_ci(scores, rng, B)
    verdict = gate(lo, bar)

    print(f"  per-item score:")
    for rec, s in zip(dataset, scores):
        print("    " + show(rec, s))
    print(f"  n = {len(dataset)}   mean score = {mean:.2f}")
    print(f"  bootstrap 95% CI (B={B} resamples, seed={seed}): [{lo:.2f}, {hi:.2f}]")
    print(f"  gate: release bar = {bar:.2f} ; CI_lo = {lo:.2f} -> "
          f"{lo:.2f} {'>=' if lo >= bar else '<'} {bar:.2f} -> {verdict}")
    return {"name": name, "mean": mean, "lo": lo, "hi": hi,
            "bar": bar, "verdict": verdict}


def _show_qa(rec, s):
    return f"{rec['q']:<20} gold={rec['gold']:<11} cand={rec['cand']:<16} -> {int(s)}"


def _show_traj(rec, s):
    g = "[" + ",".join(rec["gold"]) + "]"
    p = "[" + ",".join(rec["pred"]) + "]"
    return f"{rec['task']:<19} gold={g:<34} pred={p:<27} -> {s:.2f}"

## Run 1: QA, scored with exact match

The strict scorer. Watch item 6: `"Washington, D.C."` does not equal `"Washington"` after normalization, so it scores 0 even though a human would call it correct.

In [ ]:
line = "=" * 72
dash = "-" * 72
print(line)
print("PART 15 - A TINY EVAL HARNESS: dataset + scorer + bootstrap CI + gate.")
print(line)
print("Five reusable pieces, one file:")
print("  1. a labeled DATASET          2. a SCORER (a metric OR an LLM judge)")
print("  3. a BOOTSTRAP CI (Part 6)    4. a GATE: ship iff CI_lo >= bar (Part 11)")
print("  5. a HARNESS that wires 1-4 and prints a SHIP / HOLD verdict.")
print("Swap the dataset or the scorer; everything downstream is unchanged.")

results = []

print("\n" + dash)
print("RUN 1 - RAG-style QA eval (nod to RAG Part 8) | scorer = exact match")
print(dash)
results.append(run_eval("QA / exact-match", QA, exact_match, bar=0.35,
                        show=_show_qa))
print("  read: 0.60 looks like a win, but on n=10 the CI floor (0.30) sits")
print("        below the 0.35 bar, and strict matching failed 'Washington,")
print("        D.C.' -- a correct answer. HOLD, and reach for a better scorer.")

## Run 2: same QA data, same gate, scored with the mock judge

Nothing changes except the scorer. The judge accepts `"Washington, D.C."` for `"Washington"`, so the mean rises from 0.60 to 0.70 and the CI floor rises from 0.30 to 0.40, clearing the same 0.35 bar that Run 1 missed.

In [ ]:
print("\n" + dash)
print("RUN 2 - SAME QA dataset, SAME gate | scorer = mock LLM judge (Part 4)")
print(dash)
results.append(run_eval("QA / judge", QA, mock_judge, bar=0.35,
                        show=_show_qa))
print("  read: the judge accepts 'Washington, D.C.' for 'Washington', so the")
print("        mean rises to 0.70 and the CI floor to 0.40 -- now above 0.35.")
print("        Same data, same bar: swapping the scorer flips HOLD into SHIP.")

## Run 3: agent trajectories, scored with step-level credit

A different domain entirely, same harness. `trajectory_match` gives partial credit per task, the bootstrap CI still wraps around the mean, and the same gate logic applies, just with its own release bar.

In [ ]:
print("\n" + dash)
print("RUN 3 - agent-trajectory eval (nod to Part 14 / Agents Part 17)")
print(dash)
results.append(run_eval("agent / trajectory", TRAJ, trajectory_match, bar=0.40,
                        show=_show_traj))
print("  read: step-level credit averages to 0.70; the CI floor (0.47) clears")
print("        the 0.40 bar. The identical harness gates a different domain.")

## Summary: one harness, three evals

The same five pieces, applied to two datasets and three scorers, produce three independent SHIP / HOLD verdicts. The gate never reads the mean directly; it reads the CI lower bound. That single discipline, `CI_lo` versus a bar, is the whole series in one line.

In [ ]:
print("\n" + line)
print("SUMMARY - one harness, three evals")
print(line)
print(f"  {'eval':<20}{'mean':>6}{'95% CI':>16}{'bar':>7}{'verdict':>9}")
for r in results:
    ci = f"[{r['lo']:.2f}, {r['hi']:.2f}]"
    print(f"  {r['name']:<20}{r['mean']:>6.2f}{ci:>16}{r['bar']:>7.2f}"
          f"{r['verdict']:>9}")
print("  The gate reads the interval, never the point estimate. That single")
print("  discipline -- CI_lo vs a bar -- is the whole series in one line.")
print(line)